# Model Training & Evaluation
## AI Impact on Jobs & Layoff Risk Analysis

**Author:** Sharma Saurabh (250863131007)
**Course:** BE05000231 - Python for Data Science (GTU Semester-5)
**Month:** 3 - Modeling Phase

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import config
from src.model_training import (
    initialize_models, train_all_models, compare_models, save_best_model
)
from src.model_evaluation import (
    evaluate_model, evaluate_all_models, save_evaluation_report
)
from src.prediction import run_predictions

print('All imports successful!')

## 1. Load Preprocessed Data

In [ ]:
train_data = pd.read_csv(config.TRAIN_DATA_PATH)
test_data = pd.read_csv(config.TEST_DATA_PATH)

X_train = train_data.drop(columns=[config.TARGET_COLUMN])
y_train = train_data[config.TARGET_COLUMN]
X_test = test_data.drop(columns=[config.TARGET_COLUMN])
y_test = test_data[config.TARGET_COLUMN]

print(f'Training set: {X_train.shape}')
print(f'Testing set: {X_test.shape}')

## 2. Initialize Models

In [ ]:
models = initialize_models()
print(f'\nModels initialized: {list(models.keys())}')

## 3. Train All Models

In [ ]:
trained_models = train_all_models(models, X_train, y_train)

print('\nTrained Models:')
for name, metadata in trained_models.items():
    print(f'  {name}: Accuracy={metadata["train_accuracy"]:.4f}, Time={metadata["training_time"]:.2f}s')

## 4. Compare Models

In [ ]:
comparison_df = compare_models(trained_models, X_test, y_test)
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models_names = list(trained_models.keys())
train_accs = [trained_models[m]['train_accuracy'] for m in models_names]
test_accs = [accuracy_score(y_test, trained_models[m]['model'].predict(X_test)) for m in models_names]

x = np.arange(len(models_names))
width = 0.35

bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy', color='#3498db')
bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy', color='#e74c3c')

ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models_names, rotation=15)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Evaluate Best Model

In [ ]:
best_model_name = save_best_model(trained_models, X_test, y_test, config.MODELS_OUTPUT_DIR)
best_model = trained_models[best_model_name]['model']
print(f'\nBest Model: {best_model_name}')

In [ ]:
metrics = evaluate_model(best_model, X_test, y_test, best_model_name, config.CHARTS_OUTPUT_DIR)
print(f'\nFinal Metrics:')
print(f'  Accuracy:  {metrics["accuracy"]:.4f}')
print(f'  Precision: {metrics["precision"]:.4f}')
print(f'  Recall:    {metrics["recall"]:.4f}')
print(f'  F1-Score:  {metrics["f1_score"]:.4f}')

## 6. Make Predictions

In [ ]:
scaler = joblib.load(config.SCALER_PATH)
encoders = joblib.load(config.ENCODER_PATH)

predictions_df = run_predictions(best_model, scaler, encoders, X_test, y_test, config)
predictions_df.head(10)

## 7. Save Evaluation Report

In [ ]:
all_results = {best_model_name: metrics}
save_evaluation_report(all_results, config.EVALUATION_REPORT_PATH)
print(f'\nReport saved to: {config.EVALUATION_REPORT_PATH}')

## Modeling Summary

- **Models Trained:** Logistic Regression, Decision Tree, Random Forest
- **Best Model:** Selected based on test accuracy
- **Outputs:**
  - Best model: `outputs/models/best_model.pkl`
  - Evaluation report: `outputs/reports/evaluation_report.txt`
  - Predictions: `outputs/predictions/predictions.csv`
  - Charts: `outputs/charts/`